In [2]:
# Fabric notebook parameter. The tumbling window trigger passes its window start
# as a full ISO timestamp ("2026-08-25T02:00:00Z"), so only the date part is
# usable here.
#
# This cell must stay tagged `parameters` — Fabric injects the pipeline's value
# in a new cell directly below it, so this is a default, not a constant.
run_date = None

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 9, Finished, Available, Finished, False)

# 03 — Gold Transformation (Star Schema Build)

Driver notebook for Chapter 5. Reads `silver_job_postings` and builds the Gold
star schema: four dimensions, one fact table, one bridge table.

Order matters — the fact table resolves surrogate keys by joining against the
dimensions, so every dimension must be current before `build_fact_and_bridge`
runs:

1. `dim_date` — generated once for a wide range, no incremental logic
2. `dim_skill` — rebuilt from the curated taxonomy
3. `dim_location` — Type 1 upsert from Silver
4. `dim_company` — **SCD Type 2** upsert from Silver
5. `fact_job_postings` + `bridge_job_skill` — built, then MERGEd

All transformation logic lives in `transformation/dim_builders.py` and
`transformation/silver_to_gold.py`. This notebook only orchestrates and
inspects — no business logic below.

In [3]:
# Environment bootstrap. A Fabric notebook has neither the repo root on
# sys.path nor as its working directory, so relative paths like
# "config/config.yaml" cannot resolve there. Detecting the OneLake mount keeps
# one notebook working in both places instead of maintaining two copies.
import os
import sys

CODE_ROOT = "/lakehouse/default/Files/code"
IN_FABRIC = os.path.isdir(CODE_ROOT)

if IN_FABRIC and CODE_ROOT not in sys.path:
    sys.path.append(CODE_ROOT)

CONFIG_DIR = f"{CODE_ROOT}/config" if IN_FABRIC else "config"

# In Fabric, secrets come from the workspace rather than a gitignored .env.
# Set them here for a trial run; Chapter 8 replaces this with Key Vault.
if IN_FABRIC:
    os.environ.setdefault("ADZUNA_APP_ID", "<your-adzuna-app-id>")
    os.environ.setdefault("ADZUNA_APP_KEY", "<your-adzuna-app-key>")
    os.environ.setdefault("JOOBLE_API_KEY", "<your-jooble-api-key>")
    os.environ.setdefault("APP_ENV", "fabric")

print("Running in Fabric" if IN_FABRIC else "Running locally")

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 10, Finished, Available, Finished, False)

Running in Fabric


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import date

from delta.tables import DeltaTable

from transformation.dim_builders import (
    generate_dim_date,
    upsert_dim_company_scd2,
    upsert_dim_location,
    upsert_dim_skill,
)
from transformation.silver_to_gold import (
    build_fact_and_bridge,
    merge_fact_table,
    optimize_fact_table,
    vacuum_fact_table,
)
from utils.config_loader import load_config
from utils.logger import setup_logging, get_logger

setup_logging(config_path=f"{CONFIG_DIR}/logging_config.yaml")
logger = get_logger(__name__)

# Fabric provides a Delta-enabled session already; getOrCreate() returns it.
spark = SparkSession.builder.appName("gold_transformation").getOrCreate()

app_config = load_config(
    config_path=f"{CONFIG_DIR}/config.yaml",
    sources_path=f"{CONFIG_DIR}/sources.yaml",
)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 11, Finished, Available, Finished, False)

# Distribute the code to Spark executors

In [5]:
# Ship the code to the EXECUTORS. sys.path above only affects the driver, but
# Spark's Python workers are separate processes with their own interpreter and
# path. cloudpickle serializes module-level UDFs by reference ("import
# transformation.salary_parser, get this attribute"), so every worker has to be
# able to import the package or deserialization fails with ModuleNotFoundError
# — surfacing at the first action (.count(), .show()), not where the UDF was
# defined. addPyFile distributes the zip and adds it to each worker's path.
if IN_FABRIC:
    import shutil

    PKG_ZIP = shutil.make_archive("/tmp/jma_code", "zip", CODE_ROOT)
    try:
        spark.sparkContext.addPyFile(PKG_ZIP)
        print("Shipped", PKG_ZIP, "to executors")
    except Exception as exc:
        # Re-running in the same session hits a name collision, which is
        # harmless: the file is already registered from the first call.
        print("addPyFile skipped:", exc)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 12, Finished, Available, Finished, False)

Shipped /tmp/jma_code.zip to executors


> **Running this locally instead of in Fabric:** a Fabric notebook session has
> Delta wired into the Spark catalog already. A plain local `getOrCreate()` does
> not, and every `DeltaTable` call below fails with `Delta is not enabled`.
> Build the session like this instead:
>
> ```python
> from delta import configure_spark_with_delta_pip
>
> builder = (
>     SparkSession.builder.appName("gold_transformation")
>     .master("local[2]")
>     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
>     .config("spark.sql.catalog.spark_catalog",
>             "org.apache.spark.sql.delta.catalog.DeltaCatalog")
> )
> spark = configure_spark_with_delta_pip(builder).getOrCreate()
> ```
>
> The first run downloads the Delta jars, so it needs network access.

## Gold table paths

Same `Tables/` convention as notebooks 01 and 02 — in Fabric these resolve
inside the Lakehouse; locally they are plain directories relative to the repo
root.

In [6]:
if IN_FABRIC:
    PREFIX = "Tables/"
else:
    PREFIX = "data/delta/"

SILVER_PATH = f"{PREFIX}silver_job_postings"

DIM_DATE_PATH = f"{PREFIX}dim_date"
DIM_SKILL_PATH = f"{PREFIX}dim_skill"
DIM_LOCATION_PATH = f"{PREFIX}dim_location"
DIM_COMPANY_PATH = f"{PREFIX}dim_company"
FACT_PATH = f"{PREFIX}fact_job_postings"
BRIDGE_PATH = f"{PREFIX}bridge_job_skill"

# The run date stamps SCD2 effective dates. Parameterised rather than hardcoded
# to date.today() so a backfill can replay a historical day and still produce
# correct effective_start_date / effective_end_date values.
RUN_DATE = date.fromisoformat(run_date[:10]) if run_date else date.today()
print("Run date:", RUN_DATE)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 13, Finished, Available, Finished, False)

Run date: 2026-08-28


## Step 1 — `dim_date`

Static dimension: generated once over a wide range rather than incrementally.
Regenerating is cheap, but there is no reason to rewrite it every run, so this
cell is a no-op once the table exists.

In [7]:
if DeltaTable.isDeltaTable(spark, DIM_DATE_PATH):
    print("dim_date already exists — skipping generation")
else:
    dim_date_df = generate_dim_date(spark, date(2020, 1, 1), date(2030, 12, 31))
    dim_date_df.write.format("delta").mode("overwrite").save(DIM_DATE_PATH)
    print("Generated dim_date rows:", dim_date_df.count())

spark.read.format("delta").load(DIM_DATE_PATH).orderBy("date_key").show(5)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 14, Finished, Available, Finished, False)

dim_date already exists — skipping generation
+--------+----------+----+-----+-------+---------+
|date_key| full_date|year|month|quarter| day_name|
+--------+----------+----+-----+-------+---------+
|20200101|2020-01-01|2020|    1|      1|Wednesday|
|20200102|2020-01-02|2020|    1|      1| Thursday|
|20200103|2020-01-03|2020|    1|      1|   Friday|
|20200104|2020-01-04|2020|    1|      1| Saturday|
|20200105|2020-01-05|2020|    1|      1|   Sunday|
+--------+----------+----+-----+-------+---------+
only showing top 5 rows



## Step 2 — `dim_skill`

Rebuilt in full from `SKILL_TAXONOMY`. The taxonomy is a curated constant, not
data arriving from upstream, so a full overwrite is simpler and safer than an
upsert — a skill removed from the taxonomy should disappear from the dimension.

In [8]:
upsert_dim_skill(spark, DIM_SKILL_PATH)

dim_skill = spark.read.format("delta").load(DIM_SKILL_PATH)
print("dim_skill rows:", dim_skill.count())
dim_skill.groupBy("skill_category").count().orderBy(F.desc("count")).show()

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 15, Finished, Available, Finished, False)

2026-08-28 22:20:58 | INFO     | run_id=5433128a | transformation.dim_builders | Rebuilt dim_skill with 20 rows
dim_skill rows: 20
+--------------------+-----+
|      skill_category|count|
+--------------------+-----+
|    Data Engineering|    4|
|Programming Language|    4|
|               Cloud|    3|
|    BI/Visualization|    2|
|              DevOps|    2|
|        Data Science|    1|
|           Cloud/ETL|    1|
|      Data Warehouse|    1|
|           Streaming|    1|
|       Orchestration|    1|
+--------------------+-----+



## Step 3 — Read Silver

The single input to the whole Gold build. Everything downstream derives from
this DataFrame.

In [9]:
silver_df = spark.read.format("delta").load(SILVER_PATH)

print("Silver rows:", silver_df.count())
print("Distinct companies:", silver_df.select("company").distinct().count())
print("Distinct countries:", silver_df.select("canonical_country").distinct().count())
silver_df.printSchema()

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 16, Finished, Available, Finished, False)

Silver rows: 697
Distinct companies: 282
Distinct countries: 5
root
 |-- source: string (nullable = true)
 |-- source_job_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- company: string (nullable = true)
 |-- location_raw: string (nullable = true)
 |-- country: string (nullable = true)
 |-- description: string (nullable = true)
 |-- salary_min: double (nullable = true)
 |-- salary_max: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- remote: boolean (nullable = true)
 |-- posted_date: string (nullable = true)
 |-- url: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- raw_payload: string (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- canonical_country: string (nullable = true)
 |-- region: string (nullable = true)
 |-- final_salary_min: double (nullable = true)
 |-- final_salary_max: double (nullable = true)
 |-- final_currency: string (nullable = true)
 

## Step 4 — `dim_location` (SCD Type 1)

New `(canonical_country, region)` pairs are inserted; existing ones are left
alone. No history is kept — a country's canonical region changing is a data
correction, not a business fact anyone will ask to see historically.

In [10]:
upsert_dim_location(spark, silver_df, DIM_LOCATION_PATH)

spark.read.format("delta").load(DIM_LOCATION_PATH).show(20, truncate=False)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 17, Finished, Available, Finished, False)

2026-08-28 22:21:56 | INFO     | run_id=5433128a | transformation.dim_builders | Upserted dim_location (Type 1)
+-----------------+-------------+------------+
|canonical_country|region       |location_key|
+-----------------+-------------+------------+
|United States    |North America|0           |
|United Kingdom   |Europe       |1715033395  |
|India            |South Asia   |355824741   |
|us               |NULL         |208164208   |
|gb               |NULL         |1291759266  |
+-----------------+-------------+------------+



## Step 5 — `dim_company` (SCD Type 2)

The only Type 2 dimension in the model. `size_bucket` is derived from how many
postings a company has and genuinely evolves over the observation window —
"has this company's hiring scale changed" is a real question the project's
objectives imply.

On each run, per company:

- **new company** → insert a current row
- **`size_bucket` changed** → close the old row (`is_current = false`,
  `effective_end_date = RUN_DATE`) and insert a new current row with a new
  surrogate key
- **unchanged** → leave the existing row untouched

In [11]:
upsert_dim_company_scd2(
    spark=spark,
    silver_df=silver_df,
    table_path=DIM_COMPANY_PATH,
    run_date=RUN_DATE,
)

dim_company = spark.read.format("delta").load(DIM_COMPANY_PATH)
print("dim_company total rows:", dim_company.count())
print("dim_company current rows:", dim_company.filter(F.col("is_current")).count())
dim_company.orderBy("company_natural_key", "effective_start_date").show(20, truncate=False)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 18, Finished, Available, Finished, False)

2026-08-28 22:22:07 | INFO     | run_id=5433128a | transformation.dim_builders | dim_company: no SCD2 changes detected this run.
dim_company total rows: 286
dim_company current rows: 281
+---------------------------------------+---------------------------------------+-----------+-----------+--------------------+------------------+----------+
|company_natural_key                    |company_name                           |size_bucket|company_key|effective_start_date|effective_end_date|is_current|
+---------------------------------------+---------------------------------------+-----------+-----------+--------------------+------------------+----------+
|A&O Shearman                           |A&O Shearman                           |Small      |-1141763588|2026-08-28          |NULL              |true      |
|AGCO                                   |AGCO                                   |Small      |96         |2026-08-23          |NULL              |true      |
|AIRBUS                     

## Step 6 — `fact_job_postings` + `bridge_job_skill`

`build_fact_and_bridge` resolves every Silver row against the dimensions above
(company via the **current** SCD2 row, location, date) and extracts skills from
the description to produce the many-to-many bridge.

The fact load is a MERGE on `source_job_id`, not an append. Silver reprocessing
means the same posting can legitimately reach Gold more than once — a late
Jooble update, or a DQ rule change releasing a previously quarantined row. An
append-only fact would double-count that posting in every KPI.

In [12]:
fact_df, bridge_df = build_fact_and_bridge(
    spark=spark,
    silver_df=silver_df,
    dim_company_path=DIM_COMPANY_PATH,
    dim_location_path=DIM_LOCATION_PATH,
    dim_date_path=DIM_DATE_PATH,
    dim_skill_path=DIM_SKILL_PATH,
)

fact_df.cache()
print("Fact rows staged:", fact_df.count())
print("Bridge rows staged:", bridge_df.count())

# Unresolved keys mean a dimension is stale or a join key is dirty. Check here
# rather than discovering NULL keys in Power BI three chapters later.
fact_df.select(
    F.sum(F.col("company_key").isNull().cast("int")).alias("null_company_key"),
    F.sum(F.col("location_key").isNull().cast("int")).alias("null_location_key"),
    F.sum(F.col("date_key").isNull().cast("int")).alias("null_date_key"),
).show()

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 19, Finished, Available, Finished, False)

Fact rows staged: 616
Bridge rows staged: 6
+----------------+-----------------+-------------+
|null_company_key|null_location_key|null_date_key|
+----------------+-----------------+-------------+
|              13|                0|            0|
+----------------+-----------------+-------------+



In [13]:
merge_fact_table(spark, fact_df, FACT_PATH)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 20, Finished, Available, Finished, False)

2026-08-28 22:22:56 | INFO     | run_id=5433128a | transformation.silver_to_gold | Merged 616 rows into fact_job_postings


`silver_to_gold.py` has no bridge helper — the bridge is a pure link table with
a composite key, so an insert-if-absent MERGE keeps it idempotent with no update
branch at all. Worth promoting into the module if Chapter 6's orchestrator ends
up calling it too.

In [14]:
if not DeltaTable.isDeltaTable(spark, BRIDGE_PATH):
    bridge_df.write.format("delta").save(BRIDGE_PATH)
    print("Initialized bridge_job_skill")
else:
    (
        DeltaTable.forPath(spark, BRIDGE_PATH).alias("target")
        .merge(
            bridge_df.alias("source"),
            "target.source_job_id = source.source_job_id "
            "AND target.skill_key = source.skill_key",
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Merged into bridge_job_skill")

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 21, Finished, Available, Finished, False)

Merged into bridge_job_skill


## Verify the star schema

Row counts across every Gold table, then the joins the Power BI semantic model
in Chapter 7 will rely on. If these return sensible results, the surrogate keys
resolved correctly end to end.

In [15]:
for name, path in [
    ("dim_date", DIM_DATE_PATH),
    ("dim_skill", DIM_SKILL_PATH),
    ("dim_location", DIM_LOCATION_PATH),
    ("dim_company", DIM_COMPANY_PATH),
    ("fact_job_postings", FACT_PATH),
    ("bridge_job_skill", BRIDGE_PATH),
]:
    count = spark.read.format("delta").load(path).count()
    print(f"{name:22} {count:>8} rows")

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 22, Finished, Available, Finished, False)

dim_date                   4018 rows
dim_skill                    20 rows
dim_location                  5 rows
dim_company                 286 rows
fact_job_postings           616 rows
bridge_job_skill              6 rows


In [16]:
fact = spark.read.format("delta").load(FACT_PATH)
bridge = spark.read.format("delta").load(BRIDGE_PATH)
dim_skill = spark.read.format("delta").load(DIM_SKILL_PATH)

# Most in-demand skills — the headline question the whole project exists to answer.
(
    bridge.join(dim_skill, "skill_key")
    .groupBy("skill_name", "skill_category")
    .count()
    .orderBy(F.desc("count"))
    .show(15, truncate=False)
)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 23, Finished, Available, Finished, False)

+----------+--------------------+-----+
|skill_name|skill_category      |count|
+----------+--------------------+-----+
|scala     |Programming Language|3    |
|aws       |Cloud               |2    |
|docker    |DevOps              |1    |
+----------+--------------------+-----+



In [17]:
dim_company = spark.read.format("delta").load(DIM_COMPANY_PATH)
dim_location = spark.read.format("delta").load(DIM_LOCATION_PATH)

# Salary by country and company size — exercises three dimension joins at once.
(
    fact.join(dim_company.filter(F.col("is_current")), "company_key")
    .join(dim_location, "location_key")
    .filter(F.col("salary_min").isNotNull())
    .groupBy("canonical_country", "size_bucket")
    .agg(
        F.count("*").alias("postings"),
        F.round(F.avg("salary_min")).alias("avg_salary_min"),
        F.round(F.avg("salary_max")).alias("avg_salary_max"),
    )
    .orderBy(F.desc("postings"))
    .show(20, truncate=False)
)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 24, Finished, Available, Finished, False)

+-----------------+-----------+--------+--------------+--------------+
|canonical_country|size_bucket|postings|avg_salary_min|avg_salary_max|
+-----------------+-----------+--------+--------------+--------------+
|us               |Small      |120     |73914.0       |86191.0       |
|gb               |Medium     |62      |33460.0       |36449.0       |
|us               |Medium     |50      |67280.0       |79206.0       |
|gb               |Large      |37      |1968.0        |57162.0       |
|gb               |Small      |31      |36062.0       |39126.0       |
|us               |Large      |20      |80160.0       |88064.0       |
|United Kingdom   |Medium     |11      |37996.0       |40052.0       |
|United States    |Small      |10      |40192.0       |70673.0       |
|United Kingdom   |Large      |9       |4044.0        |60667.0       |
|United States    |Large      |4       |146475.0      |212625.0      |
|United Kingdom   |Small      |3       |40594.0       |40594.0       |
|Unite

## Verify SCD Type 2 actually worked

Only meaningful from the **second** run onwards — the first run has nothing to
close out. After a run where some company's posting count crosses a bucket
boundary, that company should show two rows: one closed, one current.

In [18]:
history = (
    dim_company
    .groupBy("company_natural_key")
    .agg(F.count("*").alias("versions"))
    .filter(F.col("versions") > 1)
)

print("Companies with SCD2 history:", history.count())

(
    dim_company.join(history, "company_natural_key")
    .select(
        "company_natural_key", "company_key", "size_bucket",
        "effective_start_date", "effective_end_date", "is_current",
    )
    .orderBy("company_natural_key", "effective_start_date")
    .show(20, truncate=False)
)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 25, Finished, Available, Finished, False)

Companies with SCD2 history: 5
+-----------------------------+-----------+-----------+--------------------+------------------+----------+
|company_natural_key          |company_key|size_bucket|effective_start_date|effective_end_date|is_current|
+-----------------------------+-----------+-----------+--------------------+------------------+----------+
|Bechtel                      |25         |Medium     |2026-08-23          |2026-08-28        |false     |
|Bechtel                      |-1141763689|Large      |2026-08-28          |NULL              |true      |
|Department of Social Services|13         |Small      |2026-08-23          |2026-08-28        |false     |
|Department of Social Services|-1141763697|Medium     |2026-08-28          |NULL              |true      |
|Lyra Health                  |59         |Small      |2026-08-23          |2026-08-28        |false     |
|Lyra Health                  |-1141763674|Medium     |2026-08-28          |NULL              |true      |
|Plaid

In [19]:
# Invariant: exactly one current row per company. A violation means the MERGE's
# close-out branch failed, and every fact join would fan out into duplicates.
violations = (
    dim_company.filter(F.col("is_current"))
    .groupBy("company_natural_key").count()
    .filter(F.col("count") > 1)
)
assert violations.count() == 0, "SCD2 invariant violated: multiple current rows"
print("SCD2 invariant holds: exactly one current row per company")

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 26, Finished, Available, Finished, False)

SCD2 invariant holds: exactly one current row per company


## Delta maintenance — OPTIMIZE, Z-ORDER, VACUUM

Not part of the per-run pipeline. Trickling small writes into a partitioned fact
table produces the small-file problem; `OPTIMIZE` compacts them and Z-ORDER
co-locates the columns the Power BI filters hit hardest. `VACUUM` then drops
files no longer referenced by any retained version.

Both are expensive and belong on a weekly schedule — that scheduling is
Chapter 6's job. Run them manually here to see what they do.

**`VACUUM` is destructive:** it deletes the underlying files time travel depends
on. The 168-hour (7 day) default is the retention floor Delta enforces for a
reason — lowering it can break in-flight readers.

In [20]:
optimize_fact_table(spark, FACT_PATH)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 27, Finished, Available, Finished, False)

2026-08-28 22:23:45 | INFO     | run_id=5433128a | transformation.silver_to_gold | Ran OPTIMIZE + ZORDER on fact_job_postings


In [21]:
# Leaves the default 7-day retention intact, so the time travel below still works.
vacuum_fact_table(spark, FACT_PATH, retention_hours=168)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 28, Finished, Available, Finished, False)

2026-08-28 22:23:58 | INFO     | run_id=5433128a | transformation.silver_to_gold | Ran VACUUM on fact_job_postings (retention=168h)


## Time travel

Every write to a Delta table creates a new version, and the transaction log keeps
old versions addressable. That makes "what changed since yesterday's load"
answerable without maintaining a snapshot table.

In [22]:
spark.sql(f"DESCRIBE HISTORY delta.`{FACT_PATH}`").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(10, truncate=False)

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 29, Finished, Available, Finished, False)

+-------+-----------------------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation   |operationMetrics                                                                                                                                                                                                                                                   

In [23]:
# Compare the current table against its first version. versionAsOf rather than a
# hardcoded timestampAsOf date, so this cell works on a fresh table — a date
# older than the table's first commit raises an error instead of returning rows.
today_df = spark.read.format("delta").load(FACT_PATH)
first_df = spark.read.format("delta").option("versionAsOf", 0).load(FACT_PATH)

print("Row count now:", today_df.count(), "| at version 0:", first_df.count())

# The timestamp form, for reference — swap in a date the table actually spans:
# yesterday_df = (
#     spark.read.format("delta")
#     .option("timestampAsOf", "2026-07-14")
#     .load(FACT_PATH)
# )

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 30, Finished, Available, Finished, False)

Row count now: 616 | at version 0: 150


## Schema evolution

`mergeSchema` lets a new column appear simply by writing a DataFrame that has
it — no manual DDL. Declaring the change explicitly is still better practice: it
puts the intent in the transaction log and in code review, rather than leaving a
column to materialise silently on some future run.

In [24]:
# Chapter 5's schema-evolution demo. Guarded because ALTER TABLE ... ADD COLUMNS
# raises FIELDS_ALREADY_EXISTS on a second run, and this notebook now runs daily
# from the pipeline — a teaching cell must not fail the Gold stage.
existing = {f.name for f in spark.read.format("delta").load(FACT_PATH).schema.fields}

if "contract_type" not in existing:
    spark.sql(f"ALTER TABLE delta.`{FACT_PATH}` ADD COLUMNS (contract_type STRING)")
    print("Added contract_type — Delta absorbed the new column with no table rewrite")
else:
    print("contract_type already present — evolution demo applied on an earlier run")

spark.read.format("delta").load(FACT_PATH).printSchema()


StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 31, Finished, Available, Finished, False)

contract_type already present — evolution demo applied on an earlier run
root
 |-- source_job_id: string (nullable = true)
 |-- source_name: string (nullable = true)
 |-- company_key: long (nullable = true)
 |-- location_key: long (nullable = true)
 |-- date_key: long (nullable = true)
 |-- salary_min: double (nullable = true)
 |-- salary_max: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- is_remote: boolean (nullable = true)
 |-- skill_count: integer (nullable = true)
 |-- loaded_at: timestamp (nullable = true)
 |-- contract_type: string (nullable = true)



In [25]:
fact_df.unpersist()
spark.stop()

StatementMeta(, 44b81298-a392-446e-9096-b90744385349, 32, Finished, Available, Finished, False)